In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

pos  = spark.read.csv("../data/POS_CASH_balance.csv", header=True, inferSchema=True)
prev = spark.read.csv("../data/previous_application.csv", header=True, inferSchema=True)

# lịch sử từng tháng của khoản vay
pos.filter(F.col("SK_ID_PREV") == 1803195).orderBy("MONTHS_BALANCE").show(100)

# thông tin hồ sơ gốc
prev.filter(F.col("SK_ID_PREV") == 1803195).show(vertical=True)

26/10/06 16:50:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+----------+--------------+--------------+---------------------+--------------------+------+----------+
|SK_ID_PREV|SK_ID_CURR|MONTHS_BALANCE|CNT_INSTALMENT|CNT_INSTALMENT_FUTURE|NAME_CONTRACT_STATUS|SK_DPD|SK_DPD_DEF|
+----------+----------+--------------+--------------+---------------------+--------------------+------+----------+
|   1803195|    182943|           -34|          48.0|                 48.0|              Active|     0|         0|
|   1803195|    182943|           -33|          48.0|                 47.0|              Active|     0|         0|
|   1803195|    182943|           -32|          48.0|                 46.0|              Active|     0|         0|
|   1803195|    182943|           -31|          48.0|                 45.0|              Active|     0|         0|
|   1803195|    182943|           -30|          48.0|                 44.0|              Active|     0|         0|
|   1803195|    182943|           -29|          48.0|                 43.0|     

In [4]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
DATA = '../data'
bb = spark.read.csv(f'{DATA}/bureau_balance.csv', header=True, inferSchema=True)

ids = [5714486, 5716150, 5716175, 5716310, 5717214, 5717358,   # đang mở
       5715003, 5715659, 5715846]                               # đã đóng
df = bb.filter(F.col('SK_ID_BUREAU').isin(ids)).toPandas()
df = df.sort_values(['SK_ID_BUREAU', 'MONTHS_BALANCE'])         # tháng cũ → tháng 0

for k, g in df.groupby('SK_ID_BUREAU'):
    print(k, g['MONTHS_BALANCE'].min(), ' '.join(g['STATUS']))
print(bb.count())                                  # số dòng, sẽ ra ~27.3 triệu
print(bb.select('SK_ID_BUREAU').distinct().count())  # số khoản có lịch sửprint('Không có lịch sử:', set(ids) - set(df['SK_ID_BUREAU']))

5717358 -27 0 0 0 0 C C C C C C C C C C C C C C C C C C C C C C C C
Không có lịch sử: {5715846, 5715659, 5716175, 5716310, 5716150, 5714486, 5715003, 5717214}


In [6]:
bb.count()                                  # số dòng
bb.select('SK_ID_BUREAU').distinct().count()   # số khoản vay có lịch sử

817395

In [7]:
print(bb.count())                                  # số dòng, sẽ ra ~27.3 triệu
print(bb.select('SK_ID_BUREAU').distinct().count())  # số khoản có lịch sử


27299925


817395


In [8]:
b = spark.read.csv(f'{DATA}/bureau.csv', header=True, inferSchema=True)
print(b.count())

1716428


In [9]:
print(b.join(bb.select('SK_ID_BUREAU').distinct(), 'SK_ID_BUREAU').count())

774354


In [10]:
print(b.join(bb.select('SK_ID_BUREAU').distinct(), 'SK_ID_BUREAU', 'left_anti').count())

942074


In [11]:
cc   = spark.read.csv(f'{DATA}/credit_card_balance.csv', header=True, inferSchema=True)
inst = spark.read.csv(f'{DATA}/installments_payments.csv', header=True, inferSchema=True)
the = cc.select('SK_ID_PREV').distinct()
print(the.count(), the.join(inst.select('SK_ID_PREV').distinct(), 'SK_ID_PREV').count())

104307 72466


In [12]:
pos = spark.read.csv(f'{DATA}/POS_CASH_balance.csv', header=True, inferSchema=True)
vay = pos.select('SK_ID_PREV').distinct()
print(vay.count(), vay.join(inst.select('SK_ID_PREV').distinct(), 'SK_ID_PREV').count())

936325 924949


In [13]:
pid = pos.select('SK_ID_PREV').first()[0]   # lấy 1 khoản vay

print('=== POS_CASH ===')
(pos.filter(F.col('SK_ID_PREV') == pid)
    .select('MONTHS_BALANCE', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'NAME_CONTRACT_STATUS', 'SK_DPD')
    .orderBy('MONTHS_BALANCE').show(50))

print('=== INSTALLMENTS ===')
(inst.filter(F.col('SK_ID_PREV') == pid)
     .select('NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT')
     .orderBy('NUM_INSTALMENT_NUMBER').show(50))

=== POS_CASH ===


+--------------+--------------+---------------------+--------------------+------+
|MONTHS_BALANCE|CNT_INSTALMENT|CNT_INSTALMENT_FUTURE|NAME_CONTRACT_STATUS|SK_DPD|
+--------------+--------------+---------------------+--------------------+------+
|           -34|          48.0|                 48.0|              Active|     0|
|           -33|          48.0|                 47.0|              Active|     0|
|           -32|          48.0|                 46.0|              Active|     0|
|           -31|          48.0|                 45.0|              Active|     0|
|           -30|          48.0|                 44.0|              Active|     0|
|           -29|          48.0|                 43.0|              Active|     0|
|           -28|          48.0|                 42.0|              Active|     0|
|           -27|          48.0|                 41.0|              Active|     0|
|           -26|          48.0|                 40.0|              Active|     0|
|           -25|

+---------------------+---------------+------------------+--------------+-----------+
|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|
+---------------------+---------------+------------------+--------------+-----------+
|                    1|        -1005.0|           -1030.0|       22629.6|    22629.6|
|                    2|         -975.0|           -1030.0|       22629.6|    22629.6|
|                    3|         -945.0|           -1009.0|       22629.6|    22629.6|
|                    4|         -915.0|            -978.0|       22629.6|    22629.6|
|                    5|         -885.0|            -944.0|       22629.6|    22629.6|
|                    6|         -855.0|            -910.0|       22629.6|    22629.6|
|                    7|         -825.0|            -869.0|       22629.6|    22629.6|
|                    8|         -795.0|            -841.0|       22629.6|    22629.6|
|                    9|         -765.0|            -84

In [14]:
pos.filter(F.col('SK_ID_CURR') == 100585).orderBy('SK_ID_PREV', 'MONTHS_BALANCE').show(100)

+----------+----------+--------------+--------------+---------------------+--------------------+------+----------+
|SK_ID_PREV|SK_ID_CURR|MONTHS_BALANCE|CNT_INSTALMENT|CNT_INSTALMENT_FUTURE|NAME_CONTRACT_STATUS|SK_DPD|SK_DPD_DEF|
+----------+----------+--------------+--------------+---------------------+--------------------+------+----------+
|   1224560|    100585|           -79|          10.0|                 10.0|              Active|     0|         0|
|   1224560|    100585|           -78|          10.0|                  9.0|              Active|     0|         0|
|   1224560|    100585|           -77|          10.0|                  8.0|              Active|     0|         0|
|   1224560|    100585|           -76|          10.0|                  7.0|              Active|     0|         0|
|   1224560|    100585|           -75|          10.0|                  6.0|              Active|     0|         0|
|   1224560|    100585|           -74|          10.0|                  5.0|     